# MobileNetV2 on Mini-ImageNet (100 classes): TensorRT FP32 / FP16 / INT8 PTQ / INT8 QAT

Extension of `01_mobilenetv2_imagenette_results.ipynb` to a harder 100-class dataset, with
dynamic-batch TensorRT engines (1-128) and an explicit TensorRT INT8 entropy calibrator.

> **Status:** code cleaned for readability; this notebook has **not yet been re-run** after
> cleanup, so it carries no outputs. Recorded results live in notebook 01.

In [ ]:
# ---- Configuration --------------------------------------------------------
# Paths default to Kaggle; change them to run elsewhere.
INPUT_DIR = "/kaggle/input"     # read-only datasets and checkpoints
WORK_DIR = "/kaggle/working"    # ONNX files, TensorRT engines, logs
MODEL_NAME = "MobileNetV2"
DATA_DIR = f"{WORK_DIR}/miniimagenet"

## 1. Setup

In [ ]:
# Environment setup (Kaggle GPU image, Tesla T4). Install output omitted.
!pip install --upgrade --index-url https://pypi.ngc.nvidia.com nvidia-tensorrt
!sudo apt-get install -y tensorrt python3-libnvinfer-dev onnx-graphsurgeon
!pip install --no-cache-dir --index-url https://pypi.nvidia.com pytorch-quantization
!pip install numpy onnx onnxruntime onnxsim wget

In [ ]:
#TensorRt installation verification
import tensorrt as trt
print(trt.__version__)
assert trt.Builder(trt.Logger())

In [ ]:
import torch
import torch.nn as nn

import torch.backends.cudnn as cudnn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.transforms as transforms
from torchvision import models, datasets
from torch.utils.tensorboard import SummaryWriter
import onnxruntime
import onnxsim

import pytorch_quantization
from pytorch_quantization import nn as quant_nn
from pytorch_quantization import quant_modules
from pytorch_quantization import calib
from pytorch_quantization.tensor_quant import QuantDescriptor
from pytorch_quantization.nn.modules.tensor_quantizer import TensorQuantizer
from pytorch_quantization.nn.modules import quant_pooling
from tqdm import tqdm

print(pytorch_quantization.__version__)
print(onnxruntime.__version__)
print(onnxsim.__version__)

import os
import sys
import warnings
import time
import random
import numpy as np
import tarfile
import shutil
import pycuda.driver as cuda
import pycuda.autoinit
import matplotlib.pyplot as plt

## 2. Data: Mini-ImageNet
Copy the dataset, map class folders to names, and build train / val / calibration loaders (500-image calibration subset).

In [ ]:
!cp -r {INPUT_DIR}/miniimagenet/ {WORK_DIR}/

In [ ]:
%%writefile {WORK_DIR}/miniimagenet/preprocess.py
import os
import json
import glob

def rename_label(map_class, train_dir, test_dir):
    class_dir_map = {}
    id_class_map = {}

    # Ensure the file exists
    if not os.path.exists(map_class):
        raise FileNotFoundError(f"File '{map_class}' not found.")

    with open(map_class, "rb") as map_class_file:
        rows = map_class_file.readlines()
        for row in rows:
            row = row.strip()
            arr = row.decode("utf-8").split(" ")
            if len(arr) < 3:
                continue  # Skip invalid lines
            class_dir_map[arr[0]] = arr[2]
            id_class_map[int(arr[1])] = arr[2]

    # Rename class folders
    for key in class_dir_map.keys():
        train_path = os.path.join(train_dir, key)
        test_path = os.path.join(test_dir, key)
        if os.path.isdir(train_path):
            os.rename(train_path, os.path.join(train_dir, class_dir_map[key]))
        if os.path.isdir(test_path):
            os.rename(test_path, os.path.join(test_dir, class_dir_map[key]))

if __name__ == "__main__":
    BASE_PATH = f"{WORK_DIR}/miniimagenet"
    MAP_CLASS_LOC = os.path.join(BASE_PATH, "map_clsloc.txt")
    TRAIN_DATA_FOLDER = os.path.join(BASE_PATH, "train/")
    TEST_DATA_FOLDER = os.path.join(BASE_PATH, "test/")

    rename_label(MAP_CLASS_LOC, TRAIN_DATA_FOLDER, TEST_DATA_FOLDER)
    print("Class renaming completed.")

In [ ]:
!python {WORK_DIR}/miniimagenet/preprocess.py

In [ ]:
# Define main data directory
DATA_DIR = f'{WORK_DIR}/miniimagenet/' 
# Define training and validation data paths
TRAIN_DIR = os.path.join(DATA_DIR, 'train') 
VAL_DIR = os.path.join(DATA_DIR, 'test')

In [ ]:
#Performing Transformations on the dataset and defining training and validation dataloaders
torch.backends.cudnn.benchmark = True  # Enables auto-tuning for speed optimization
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),  # Random crop instead of CenterCrop
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
}

calib_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),  
    transforms.ToTensor(),  
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=data_transforms['val'])
val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'), transform=data_transforms['val'])

train_dataloader = data.DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True,pin_memory=True,num_workers=4)
val_dataloader = data.DataLoader(val_dataset, batch_size=32, shuffle=False, drop_last=False,pin_memory=True,num_workers=4)

# Load a small subset for calibration
# Select a subset of 100 images for calibration
CALIBRATION_SIZE = 500
random.seed(42)
calib_indices = random.sample(range(len(val_dataset)), CALIBRATION_SIZE)
calib_subset = torch.utils.data.Subset(val_dataset, calib_indices)
calib_dataloader = torch.utils.data.DataLoader(
    calib_subset, batch_size=32, shuffle=False, drop_last=True, pin_memory=True, num_workers=4
)

In [ ]:
# Get a batch of images
dataiter = iter(train_dataloader)
images, labels = next(dataiter)

def imshow(img):
    img = img.numpy().transpose((1, 2, 0))  # Convert from (C, H, W) to (H, W, C)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img * std + mean  # Unnormalize
    img = np.clip(img, 0, 1)  # Clip values to [0,1] to prevent incorrect display
    plt.imshow(img)
    plt.axis("off")

# Plot images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

for i in range(5):  # Only select first 5 images
    ax = axes[i]
    ax.imshow(images[i].permute(1, 2, 0).numpy())  # Convert Tensor to Image
    ax.set_title(train_dataset.classes[labels[i]])
    ax.axis("off")

plt.show()

## 3. FP32 baseline (MobileNetV2, 100-class head)

In [ ]:
#This function allows you to set the all the parameters to not have gradients, 
#allowing you to freeze the model and not undergo training during the train step. 
def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False

feature_extract = True #This varaible can be set False if you want to finetune the model by updating all the parameters. 
#--------Load Torchvision Model------------------
model = models.mobilenet_v2(pretrained=False)

set_parameter_requires_grad(model, feature_extract)
#Define a classification head for 10 classes.
#-----Mobilenetv2---------------------------
model.classifier[1] = nn.Linear(1280, 100)

# num_ftrs = model.fc.in_features
# model.fc = nn.Linear(num_ftrs, 100)
#-------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.cuda()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=5e-3, momentum=0.9, weight_decay=1e-4)  # Lower LR & weight decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

In [ ]:
ckpt = torch.load(f"{INPUT_DIR}/mobilenetv2/pytorch/default/1/best_mobilenetv2_ckpt(4)")
modified_state_dict={}
for key, val in ckpt["model_state_dict"].items():
    # Remove 'module.' from the key names
    if key.startswith('module'):
        modified_state_dict[key[7:]] = val
    else:
        modified_state_dict[key] = val

# Load the pre-trained checkpoint
model.load_state_dict(modified_state_dict)
optimizer.load_state_dict(ckpt["opt_state_dict"])

In [ ]:
# Define functions for training, evalution, saving checkpoint and train parameter setting function
def train(model, dataloader, crit, opt, epoch):
    model.train()
    total_loss = 0.0
    total_corrects = 0
    total_samples = 0

    for data, labels in dataloader:
        data, labels = data.to(device), labels.to(device)

        # Zero the parameter gradients
        opt.zero_grad()

        # Forward pass
        out = model(data)
        loss = crit(out, labels)

        # Backward pass and optimize
        loss.backward()
        opt.step()
        scheduler.step()

        # Accumulate loss
        total_loss += loss.item()

        # Calculate number of correct predictions
        _, preds = torch.max(out, 1)
        total_corrects += (preds == labels).sum().item()
        total_samples += labels.size(0)

    # Calculate average loss and accuracy for the epoch
    avg_loss = total_loss / total_samples
    avg_acc = total_corrects / total_samples

    return avg_loss, avg_acc

    

def evaluate(model, dataloader, crit, epoch, dtype='fp32'):
    model.eval()
    total_loss = 0.0
    total_corrects = 0
    total_samples = 0

    with torch.no_grad():  # Disable gradient computation
        for data, labels in dataloader:
            # Ensure data type consistency
            if dtype == 'fp16':
                data, labels = data.half().to(device), labels.to(device)
            else:
                data, labels = data.to(device), labels.to(device)

            # Forward pass
            out = model(data)
            loss = crit(out, labels)

            # Accumulate loss
            total_loss += loss.item()

            # Calculate number of correct predictions
            _, preds = torch.max(out, 1)
            total_corrects += (preds == labels).sum().item()
            total_samples += labels.size(0)

    # Calculate average loss and accuracy for the entire dataset
    avg_loss = total_loss / total_samples
    avg_acc = total_corrects / total_samples

    return avg_loss, avg_acc

def save_checkpoint(state, ckpt_path="checkpoint.pth"):
    torch.save(state, ckpt_path)
    print("Checkpoint saved")
    
# Helper function to benchmark the model
cudnn.benchmark = True
import time
import numpy as np
import torch.backends.cudnn as cudnn  # Add missing import

def benchmark(model, input_shape=(1024, 3, 224, 224), dtype='fp32', nwarmup=50, nruns=100):
    model.eval()
    input_data = torch.randn(input_shape).to(device)
    
    if dtype == 'fp16':
        input_data = input_data.half()

    print("Warm up ...")
    with torch.no_grad():
        for _ in range(nwarmup):
            _ = model(input_data)

    torch.cuda.synchronize()
    
    print("Start timing ...")
    timings = []
    with torch.no_grad():
        for i in range(nruns):
            start_time = time.time()
            output = model(input_data)
            torch.cuda.synchronize()
            end_time = time.time()
            timings.append(end_time - start_time)

    print("Input shape:", input_data.size())
    print("Output features size:", output.size())
    print('Average batch time: %.2f ms' % (np.mean(timings) * 1000))
    print('Average throughput: %.2f images/second' % (input_shape[0] / np.mean(timings)))

# EarlyStopping class for training monitoring
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_accuracy):
        if self.best_score is None:
            self.best_score = val_accuracy
        elif val_accuracy < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_accuracy
            self.counter = 0

In [ ]:
num_epochs = 30
best_acc = 0.0
log_path = f'{WORK_DIR}/training_log_mobilenetv2.txt'
patience = 8  # Stop training if no improvement for 5 epochs
early_stopping = EarlyStopping(patience=patience, min_delta=0.001)

# Open log file
with open(log_path, 'w') as log_file:
    log_file.write("Epoch, Train Loss, Train Acc, Val Loss, Val Acc \n")

    for epoch in range(num_epochs):
    
        train_loss, train_acc = train(model, train_dataloader, criterion, optimizer, epoch)
        test_loss, test_acc = evaluate(model, val_dataloader, criterion, epoch)
        
        # Adjust learning rate
        scheduler.step()
    
        print(f"Epoch {epoch + 1} - Training Loss: {train_loss:.4f}, Training Accuracy: {train_acc * 100:.2f}%")
        print(f"Epoch {epoch + 1} - Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc * 100:.2f}%")
        # Log metrics to file
        log_file.write(f"{epoch + 1}, {train_loss:.4f}, {train_acc:.4f}, {test_loss:.4f}, {test_acc:.4f}\n")

    
        # Save the best model
        if test_acc > best_acc:
            best_acc = test_acc
            save_checkpoint({'epoch': epoch + 1,
                             'model_state_dict': model.state_dict(),
                             'acc': test_acc,
                             'opt_state_dict': optimizer.state_dict()},
                            ckpt_path=f"{WORK_DIR}/best_mobilenetv2_ckpt")
            print(f"Best model saved at epoch {epoch + 1}")
            
        # Early stopping
        early_stopping(test_acc)
        if early_stopping.early_stop:
            print("Early stopping triggered. Stopping training.")
            break

print("Training complete. Best Accuracy: {:.2f}%".format(best_acc * 100))

In [ ]:
# Evaluate and benchmark the performance of the baseline Pytorch model (FP32 Model)
test_loss, test_acc = evaluate(model, val_dataloader, criterion, 0)
print(f"{MODEL_NAME} PyTorch FP32 accuracy: {100 * test_acc:.2f}%")

benchmark(model, input_shape=(1, 3, 224, 224))
benchmark(model, input_shape=(32, 3, 224, 224))

## 4. TensorRT INT8 entropy calibrator

In [ ]:
class Int8EntropyCalibrator(trt.IInt8EntropyCalibrator2):
    def __init__(self, calib_dataloader, cache_file=f"{WORK_DIR}/calibration_data.cache"):
        super().__init__()
        self.calib_dataloader = iter(calib_dataloader)
        self.cache_file = cache_file
        self.batch_size = 32  # Must match dataloader batch_size
        self.data_shape = (self.batch_size, 3, 224, 224)  # Ensure TensorRT-compatible shape
        self.device_input = cuda.mem_alloc(trt.volume(self.data_shape) * trt.float32.itemsize)

    def get_batch_size(self):
        return self.batch_size

    def get_batch(self, names):
        """Fetches a batch of data for calibration."""
        try:
            images, _ = next(self.calib_dataloader)  # Get next batch
            batch = images.to('cpu').numpy().astype(np.float32)  # Ensure float32 precision
            cuda.memcpy_htod(self.device_input, batch)  # Copy to CUDA device
            return [self.device_input]
        except StopIteration:
            return None  # No more data

    def read_calibration_cache(self):
        """Loads cached calibration data if available to avoid re-calibration."""
        if os.path.exists(self.cache_file):
            print(f"Loading cached calibration file: {self.cache_file}")
            with open(self.cache_file, "rb") as f:
                return f.read()
        return None

    def write_calibration_cache(self, cache):
        """Writes calibration data to cache for reuse."""
        print(f"Saving calibration data to {self.cache_file}")
        with open(self.cache_file, "wb") as f:
            f.write(cache)

In [ ]:
# Initialize TensorRT logger
logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)

# Create network and config
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
config = builder.create_builder_config()
config.set_flag(trt.BuilderFlag.INT8)  # Enable INT8 Precision

# Set up calibration
calibrator = Int8EntropyCalibrator(calib_dataloader)  # Pass your calibration dataloader
config.int8_calibrator = calibrator

## 5. ONNX export and TensorRT engines (FP32, FP16, INT8 PTQ)

In [ ]:
# Exporting to Onnx
dummy_input = torch.randn(1, 3, 224, 224, device='cuda')
input_names = [ "actual_input_1" ]
output_names = [ "output1" ]
# Export the model to ONNX with normalized input
torch.onnx.export(
    model.eval(), 
    dummy_input,  # Pass the dummy input for export
    f"{WORK_DIR}/mobilenetv2_base.onnx",
    #f"{WORK_DIR}/mobilenetv2_base.onnx",
    verbose=True,
    opset_version=13,
    do_constant_folding=False,
    input_names=input_names,
    output_names=output_names,
    dynamic_axes={"actual_input_1": {0: "batch_size"}}  # Allow dynamic batch sizes
)

print("✅ ONNX Model Exported with Correct Normalization!")

In [ ]:
!python -m onnxsim {WORK_DIR}/mobilenetv2_base.onnx {WORK_DIR}/mobilenetv2_base_simplified.onnx

In [ ]:
!/usr/src/tensorrt/bin/trtexec --onnx={WORK_DIR}/mobilenetv2_base_simplified.onnx \
  --saveEngine={WORK_DIR}/mobilenetv2_base_fp32.trt --useCudaGraph --useSpinWait --iterations=100 \
  --minShapes=actual_input_1:1x3x224x224 \
  --optShapes=actual_input_1:32x3x224x224 \
  --maxShapes=actual_input_1:128x3x224x224

In [ ]:
!/usr/src/tensorrt/bin/trtexec --onnx={WORK_DIR}/mobilenetv2_base_simplified.onnx \
  --saveEngine={WORK_DIR}/mobilenetv2_base_fp16.trt --fp16 --useCudaGraph --useSpinWait --iterations=100 \
  --minShapes=actual_input_1:1x3x224x224 \
  --optShapes=actual_input_1:32x3x224x224 \
  --maxShapes=actual_input_1:128x3x224x224

In [ ]:
!/usr/src/tensorrt/bin/trtexec --onnx={WORK_DIR}/mobilenetv2_base_simplified.onnx \
  --saveEngine={WORK_DIR}/mobilenetv2_int8.trt --int8 --calib={WORK_DIR}/calibration_data.cache --useCudaGraph --useSpinWait --iterations=100 \
  --minShapes=actual_input_1:1x3x224x224 \
  --optShapes=actual_input_1:32x3x224x224 \
  --maxShapes=actual_input_1:128x3x224x224

## 6. INT8 with pytorch-quantization (fake-quant PTQ)
Per-tensor histogram calibration for activations, per-channel max for weights.

In [ ]:
# ## Force per tensor quantization for activations and per channel quantization for weight for onnx runtime
quant_desc_input = QuantDescriptor(num_bits=8,calib_method="histogram", axis=None)
quant_nn.QuantConv2d.set_default_quant_desc_input(quant_desc_input)
quant_nn.QuantConvTranspose2d.set_default_quant_desc_input(quant_desc_input)
quant_nn.QuantLinear.set_default_quant_desc_input(quant_desc_input)

quant_desc_weight = QuantDescriptor(num_bits=8,calib_method="max", axis=0)
quant_nn.QuantConv2d.set_default_quant_desc_weight(quant_desc_weight)
quant_nn.QuantConvTranspose2d.set_default_quant_desc_weight(quant_desc_weight)
quant_nn.QuantLinear.set_default_quant_desc_weight(quant_desc_weight)

In [ ]:
#This function allows you to set the all the parameters to not have gradients, 
#allowing you to freeze the model and not undergo training during the train step. 

quant_modules.initialize()
def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False

feature_extract = True #This varaible can be set False if you want to finetune the model by updating all the parameters. 
#--------Load Torchvision Model------------------
q_model = models.mobilenet_v2(pretrained=False)

set_parameter_requires_grad(q_model, feature_extract)
#Define a classification head for 10 classes.
#-----Mobilenetv2---------------------------
q_model.classifier[1] = nn.Linear(1280, 100)

# num_ftrs = q_model.fc.in_features
# q_model.fc = nn.Linear(num_ftrs, 100)
#-------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
q_model = q_model.cuda()

In [ ]:
ckpt = torch.load(f"{INPUT_DIR}/imageclassification/pytorch/default/1/mobilenetv2(1).pth")
modified_state_dict={}
for key, val in ckpt["model_state_dict"].items():
    # Remove 'module.' from the key names
    if key.startswith('module'):
        modified_state_dict[key[7:]] = val
    else:
        modified_state_dict[key] = val

# Load the pre-trained checkpoint
q_model.load_state_dict(modified_state_dict)
optimizer.load_state_dict(ckpt["opt_state_dict"])

In [ ]:
from pytorch_quantization.calib import MaxCalibrator, HistogramCalibrator

def collect_stats(model, data_loader, num_batches):
    """Feed data to the network and collect statistics."""

    # ✅ Enable calibration mode (disable quantization)
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                module.disable_quant()  # Disable quantization
                module.enable_calib()   # Enable calibration
            else:
                module.disable()  # If no calibrator, fully disable

    # ✅ Feed data to the model for collecting statistics
    for i, (image, _) in tqdm(enumerate(data_loader), total=num_batches):
        model(image.cuda())  # Ensure image is on CUDA
        if i >= num_batches - 1:  # Ensure correct number of batches
            break

    # ✅ Disable calibration mode (enable quantization)
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                module.enable_quant()   # Enable quantization
                module.disable_calib()  # Disable calibration
            else:
                module.enable()  # If no calibrator, enable quantization

def compute_amax(model, **kwargs):
    """Load calibration result and apply amax values."""
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer) and module._calibrator is not None:
            if isinstance(module._calibrator, MaxCalibrator):
                module.load_calib_amax()  # ✅ Max Calibration (default)
            else:
                module.load_calib_amax(**kwargs)  # ✅ Other methods like histogram
    
    return model  # ✅ Return model explicitly

In [ ]:
print("📊 Running Compute Amax for Calibration...")

# ✅ Move model to CUDA before calibration
q_model.cuda()
q_model.eval()

# ✅ Run calibration
with torch.no_grad():
    collect_stats(q_model, data_loader=train_dataloader, num_batches=32)

# ✅ Compute amax with histogram for activations, max for weights
with torch.no_grad():
    q_model = compute_amax(q_model, method="max", percentile=99.9)

In [ ]:
#Evaluate model upon calibration
with torch.no_grad():
    test_loss, test_acc = evaluate(q_model, val_dataloader, criterion, 0)
    print(f"{MODEL_NAME} fake-quant INT8 accuracy: {100 * test_acc:.2f}%")

In [ ]:
# Set static member of TensorQuantizer to use Pytorch’s own fake quantization functions
quant_nn.TensorQuantizer.use_fb_fake_quant = True

# Exporting to Onnx
dummy_input = torch.randn(1, 3, 224, 224, device='cuda')
input_names = [ "actual_input_1" ]
output_names = [ "output1" ]
torch.onnx.export(
    q_model.eval(),
    dummy_input,
    f"{WORK_DIR}/mobilenetv2_ptq.onnx",
    verbose=False,
    opset_version=13,
    do_constant_folding = False,
    input_names = input_names,
    output_names = output_names,)

# Converting ONNX model to TRT
!/usr/src/tensorrt/bin/trtexec --onnx={WORK_DIR}/mobilenetv2_ptq.onnx --int8 --useSpinWait --saveEngine={WORK_DIR}/mobilenetv2_ptq.trt

## 7. Quantization-aware training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, verbose=True)
# 

In [ ]:
num_epochs = 5
best_acc = 0.0
log_path = f'{WORK_DIR}/training_log_mobilenetv2qat.txt'
patience = 10  # Stop training if no improvement for 5 epochs
early_stopping = EarlyStopping(patience=patience)

# Open log file
with open(log_path, 'w') as log_file:
    log_file.write("Epoch, Train Loss, Train Acc, Val Loss, Val Acc \n")

    for epoch in range(num_epochs):
    
        train_loss, train_acc = train(q_model, train_dataloader, criterion, optimizer, epoch)
        test_loss, test_acc = evaluate(q_model, val_dataloader, criterion, epoch)
        
        # Adjust learning rate
        # scheduler.step(test_loss)
    
        print(f"Epoch {epoch + 1} - Training Loss: {train_loss:.4f}, Training Accuracy: {train_acc * 100:.2f}%")
        print(f"Epoch {epoch + 1} - Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc * 100:.2f}%")
        # Log metrics to file
        log_file.write(f"{epoch + 1}, {train_loss:.4f}, {train_acc:.4f}, {test_loss:.4f}, {test_acc:.4f}\n")

    
        # Save the best model
        if test_acc > best_acc:
            best_acc = test_acc
            save_checkpoint({'epoch': epoch + 1,
                             'model_state_dict': q_model.state_dict(),
                             'acc': test_acc,
                             'opt_state_dict': optimizer.state_dict()},
                            ckpt_path=f"{WORK_DIR}/best_mobilenetv2qat_ckpt")
            print(f"Best model saved at epoch {epoch + 1}")
            
        # Early stopping
        early_stopping(test_acc)
        if early_stopping.early_stop:
            print("Early stopping triggered. Stopping training.")
            break

print("Training complete. Best Accuracy: {:.2f}%".format(best_acc * 100))

In [ ]:
ckpt = torch.load(f"{WORK_DIR}/best_mobilenetv2qat_ckpt")
modified_state_dict={}
for key, val in ckpt["model_state_dict"].items():
    # Remove 'module.' from the key names
    if key.startswith('module'):
        modified_state_dict[key[7:]] = val
    else:
        modified_state_dict[key] = val

# Load the pre-trained checkpoint
q_model.load_state_dict(modified_state_dict)
optimizer.load_state_dict(ckpt["opt_state_dict"])

In [ ]:
#Evaluate and benchmark the performance of the QAT-TRT model (TRT INT8)
test_loss, test_acc = evaluate(q_model, val_dataloader, criterion, 0)
print(f"{MODEL_NAME} fake-quant INT8 accuracy: {100 * test_acc:.2f}%")
benchmark(q_model, input_shape=(1, 3, 224, 224))

## 8. Export the QAT model and benchmark all engines

In [ ]:
# Set static member of TensorQuantizer to use Pytorch’s own fake quantization functions
quant_nn.TensorQuantizer.use_fb_fake_quant = True
# Exporting to Onnx
dummy_input = torch.randn(1, 3, 224, 224, device='cuda')
input_names = [ "actual_input_1" ]
output_names = [ "output1" ]

# Define dynamic axes to allow flexibility in the batch size and spatial dimensions

torch.onnx.export(
    q_model.eval(),
    dummy_input,
    f"{WORK_DIR}/mobilenetv2_qat.onnx",
    # f"{WORK_DIR}/mobilenetv2_qat.onnx",
    input_names=input_names,
    output_names=output_names,
    dynamic_axes={"actual_input_1": {0: "batch_size"}},  # Allow dynamic batch sizes
    verbose=False,
    opset_version=13,
    do_constant_folding=False)

# Converting ONNX model to TRT
!/usr/src/tensorrt/bin/trtexec --onnx={WORK_DIR}/mobilenetv2_qat.onnx --useSpinWait --useCudaGraph --int8 --iterations=100 --saveEngine={WORK_DIR}/mobilenetv2_qat.trt\
--minShapes=actual_input_1:1x3x224x224 \
--optShapes=actual_input_1:32x3x224x224 \
--maxShapes=actual_input_1:128x3x224x224

In [ ]:
import numpy as np
import torch
import time
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit

def  evaluate_trt_dynamic(trt_engine_path, dataloader, criterion, batch_sizes=[1, 16, 32]):
    # Load TensorRT engine
    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    with open(trt_engine_path, "rb") as f, trt.Runtime(TRT_LOGGER) as runtime:
        engine = runtime.deserialize_cuda_engine(f.read())
    
    # Prepare accuracy, latency, and throughput metrics
    total_correct = {batch_size: 0 for batch_size in batch_sizes}
    total_samples = {batch_size: 0 for batch_size in batch_sizes}
    latencies = {batch_size: [] for batch_size in batch_sizes}
    throughputs = {}
    
    # Evaluate for each batch size
    for batch_size in batch_sizes:
        print(f"\nEvaluating for batch size: {batch_size}")

        # Create context and allocate memory based on batch size
        context = engine.create_execution_context()
        input_shape = (batch_size, 3, 224, 224)  # Adjust as needed
        context.set_binding_shape(0, input_shape)
        
        # Allocate GPU memory for inputs and outputs
        input_size = trt.volume(input_shape) * trt.float32.itemsize
        output_shape = context.get_binding_shape(1)
        output_size = trt.volume(output_shape) * trt.float32.itemsize

        d_input = cuda.mem_alloc(input_size)
        d_output = cuda.mem_alloc(output_size)
        h_output = np.empty(output_shape, dtype=np.float32)  # Host output buffer
        
        # Initialize timing
        total_time = 0.0
        total_batches = 0

        for i, (data, labels) in enumerate(dataloader):
            if data.size(0) < batch_size:
                break  # Ensure complete batches for consistency

            data = data[0:batch_size].cuda()  # Select batch size
            labels = labels[0:batch_size].cuda()

            # Transfer data to the GPU
            cuda.memcpy_htod(d_input, data.cpu().numpy().astype(np.float32))

            # Measure latency
            start_time = time.time()
            context.execute_v2(bindings=[int(d_input), int(d_output)])
            cuda.memcpy_dtoh(h_output, d_output)
            end_time = time.time()

            latency = end_time - start_time
            latencies[batch_size].append(latency)
            total_time += latency
            total_batches += 1

            # Convert output to PyTorch tensor for accuracy calculation
            output = torch.from_numpy(h_output).cuda()
            loss = criterion(output, labels)
            preds = output.argmax(dim=1)
            total_correct[batch_size] += (preds == labels).sum().item()
            total_samples[batch_size] += labels.size(0)

        # Throughput calculation
        avg_latency = np.mean(latencies[batch_size]) * 1000  # in ms
        throughput = (total_batches * batch_size) / total_time  # samples per second
        throughputs[batch_size] = throughput

        # Print results
        accuracy = total_correct[batch_size] / total_samples[batch_size]
        print(f"Batch Size: {batch_size}")
        print(f"Accuracy: {accuracy * 100:.2f}%")
        print(f"Average Latency: {avg_latency:.2f} ms")
        print(f"Throughput: {throughput:.2f} samples/second")

    return total_correct, latencies, throughputs

In [ ]:
trt_engine_path = f"{WORK_DIR}/mobilenetv2_base_fp32.trt"
for bs in (1, 32):
    evaluate_trt_dynamic(trt_engine_path, val_dataloader, criterion, batch_sizes=[bs])

In [ ]:
trt_engine_path = f"{WORK_DIR}/mobilenetv2_base_fp16.trt"
for bs in (1, 32):
    evaluate_trt_dynamic(trt_engine_path, val_dataloader, criterion, batch_sizes=[bs])

In [ ]:
trt_engine_path = f"{WORK_DIR}/mobilenetv2_int8.trt"
for bs in (1, 32):
    evaluate_trt_dynamic(trt_engine_path, val_dataloader, criterion, batch_sizes=[bs])

In [ ]:
trt_engine_path = f"{WORK_DIR}/mobilenetv2_qat.trt"
for bs in (1, 32):
    evaluate_trt_dynamic(trt_engine_path, val_dataloader, criterion, batch_sizes=[bs])

In [ ]:
for eng in ["mobilenetv2_base_fp32", "mobilenetv2_base_fp16", "mobilenetv2_int8", "mobilenetv2_qat"]:
    size_mb = os.path.getsize(f"{WORK_DIR}/{eng}.trt") / (1024 * 1024)
    print(f"{eng:>24}: {size_mb:.2f} MB")